# Shadow Agent Scan

Find the AI agents nobody registered in two public repositories (one on GitHub, one on GitLab) with **ShadowScan**, read what the evidence shows, and bring one of them under governance.

**Time:** about 30 minutes · **Needs:** Python 3.11+, git, Linux/macOS or Google Colab · **No tokens**, and nothing from the repositories is executed · **Scanner:** [`aisecnomad/Project-Nexus`](https://github.com/aisecnomad/Project-Nexus) pinned to commit [`9d4a6f1261f1`](https://github.com/aisecnomad/Project-Nexus/tree/9d4a6f1261f1daba97c2c147ca088dec78e89a46) (validated 3 Oct 2026)

| Step | You will | Min |
|---|---|---:|
| 1 | Install the pinned scanner | 3 |
| 2 | Fetch two pinned repositories and scan them | 5 |
| 3 | Read a finding: evidence → confidence → risk factors | 7 |
| 4 | Reconcile: empty inventory → stub → approve one agent → rescan → diff | 10 |
| 5 | Export a report and run the CI gate | 3 |

Run the cells in order. Everything is written to `shadow-agent-lab/` next to this notebook.

## 1 · Install

The scanner goes into its own virtual environment inside the lab folder, pinned to a reviewed commit: a floating `main` would run scanner code nobody reviewed. Windows is refused by the scanner itself; use WSL2 or Colab there.

In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import venv
from pathlib import Path

assert sys.version_info >= (3, 11), "use a Python 3.11+ kernel"
assert not sys.platform.startswith("win"), "ShadowScan refuses Windows: use WSL2, a Linux container or Google Colab"
assert shutil.which("git"), "install git first"

SCANNER_COMMIT = "9d4a6f1261f1daba97c2c147ca088dec78e89a46"  # reviewed revision this lab was validated against
LAB = Path.cwd() / "shadow-agent-lab"
for folder in ("targets", "reports", "inventory"):
    (LAB / folder).mkdir(parents=True, exist_ok=True)
PY = LAB / "scanner-env" / "bin" / "python"
if not PY.exists():
    venv.EnvBuilder(with_pip=True).create(LAB / "scanner-env")
    subprocess.run([PY, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                    f"git+https://github.com/aisecnomad/Project-Nexus.git@{SCANNER_COMMIT}"], check=True)

ANSI = re.compile(r"\x1b\[[0-9;?]*[A-Za-z]")


def shadowscan(*args):
    """Run the CLI (-q: errors only) and show its output. Exit codes: 0 complete · 2 --fail-on tripped · 3 INCOMPLETE · 1 usage error."""
    print("$ shadowscan -q", " ".join(map(str, args)))
    run = subprocess.run([PY, "-m", "shadowscan", "-q", *map(str, args)], text=True, capture_output=True,
                         env={**os.environ, "COLUMNS": "150", "NO_COLOR": "1"})
    print(ANSI.sub("", run.stdout + run.stderr).rstrip(), f"\n-> exit {run.returncode}\n")
    return run.returncode


def load(name):
    """Load reports/<name>.json and print its summary plus every connector note."""
    report = json.loads((LAB / "reports" / f"{name}.json").read_text(encoding="utf-8"))
    s = report["summary"]
    print(f"{name}: {s['total']} findings · shadow={s['shadow']} · complete={s['complete']} · {s['by_risk_level']}")
    for stat in report["stats"]:
        for note in stat["warnings"] + stat["errors"]:
            print("   note:", note)
    return report


_ = shadowscan("--version")

## 2 · Fetch and scan

**What you are looking for.** An agent is software that lets a model decide *and act*: call tools, run code, read data. In a repository it shows up as agent frameworks (LangGraph, CrewAI, OpenAI Agents SDK, …), MCP client/server configs (`.mcp.json`, `.cursor/mcp.json`), coding-agent configs (`CLAUDE.md`, `AGENTS.md`), agent manifests, infrastructure code that provisions cloud agents, and hard-coded provider keys. A **shadow agent** is one nobody registered, owns or approved. Three questions drive everything: *who owns it, what can it do, did anyone approve it?*

**The two repositories** are fetched at a pinned commit, one commit deep, with git hooks, submodules and LFS disabled, so nothing in them runs:

| Host | Repository | Why |
|---|---|---|
| GitHub | [`modelcontextprotocol/servers`](https://github.com/modelcontextprotocol/servers/tree/f46d9578190b476b3501923ea8977d899e8db2cb) | the reference MCP servers, plus a `CLAUDE.md`, a `.mcp.json` and an `AGENTS.md`: three different finding kinds in seconds |
| GitLab | [`imolko/crewmaster`](https://gitlab.com/imolko/crewmaster/-/tree/5b76c10cf7f800fcbb2591057bae646302526b00) | a multi-agent framework on LangGraph: one high-risk finding and two scanner warnings worth reading |

In [ ]:
TARGETS = {
    "github-mcp-servers": ("https://github.com/modelcontextprotocol/servers.git", "f46d9578190b476b3501923ea8977d899e8db2cb"),
    "gitlab-crewmaster": ("https://gitlab.com/imolko/crewmaster.git", "5b76c10cf7f800fcbb2591057bae646302526b00"),
}


def fetch(name, url, commit):
    """Fetch exactly one commit over HTTPS; no hooks, submodules, LFS content or prompts."""
    dest = LAB / "targets" / name
    git = ["git", "-c", "core.hooksPath=/dev/null", "-c", "submodule.recurse=false", "-c", "credential.helper="]
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0", "GIT_LFS_SKIP_SMUDGE": "1"}
    if not (dest / ".git").exists():
        dest.mkdir(parents=True)
        subprocess.run([*git, "init", "-q", dest], check=True, env=env)
        subprocess.run([*git, "-C", dest, "fetch", "-q", "--depth=1", "--no-tags", "--no-recurse-submodules", url, commit], check=True, env=env)
        subprocess.run([*git, "-C", dest, "checkout", "-q", "--detach", "FETCH_HEAD"], check=True, env=env)
    head = subprocess.check_output(["git", "-C", dest, "rev-parse", "HEAD"], text=True).strip()
    assert head == commit, f"{name}: checked out {head}, expected {commit}"
    files = sum(1 for p in dest.rglob("*") if p.is_file() and ".git" not in p.parts)
    print(f"{name:20s} {commit[:12]}  {files} files")
    return dest


PATHS = [fetch(name, url, commit) for name, (url, commit) in TARGETS.items()]

`shadowscan code PATH…` scans local checkouts; no config file, no token. `--format json --output file.json` saves the full report the rest of the lab reads (run the same command without `--format` in a terminal for the table view). There is no inventory yet, so every finding must come back with `shadow: null`: *discovered, not assessed*.

In [ ]:
rc = shadowscan("code", *PATHS, "--format", "json", "--output", LAB / "reports/discovery.json")
discovery = load("discovery")
assert rc == 0, "the scan was not complete: read the notes above"
assert all(f["shadow"] is None for f in discovery["findings"]), "no inventory was given, so shadow must be null"


def listing(report):
    for f in sorted(report["findings"], key=lambda f: -f["risk"]["score"]):
        print(f"{f['risk']['level']:8} {f['risk']['score']:3}  shadow={str(f['shadow']):5}  {f['kind']:16} {f['title'][:72]}")


print()
listing(discovery)

**Validated result (3 Oct 2026):** 12 findings, scan complete. `modelcontextprotocol/servers`: 2 `agent-config`, 1 `mcp-server`, 8 `framework-usage`. `imolko/crewmaster`: 1 `framework-usage` at *high* plus two notes (an unparseable `mkdocs.yml`, an oversize PNG); images are on the declared skip list, so the scan stays **complete**. Read the notes before the findings: a note tells you what was not assessed, exit 3 would mean the totals cannot be trusted.

## 3 · Read a finding

Every score is explainable: `risk.factors` lists each contribution, `evidence` lists every signal with its weight and file location.

In [ ]:
def show(f, evidence=5):
    print(f"{f['title']}\n  kind={f['kind']}  confidence={f['confidence']:.2f} ({f['likelihood']})  "
          f"risk={f['risk']['score']} {f['risk']['level'].upper()}  shadow={f['shadow']}  owner={f['owner']}")
    print("  factors     :", ", ".join(f"{x['id']}({x['weight']:+d})" for x in f["risk"]["factors"]))
    print("  capabilities:", f["capabilities"], " frameworks:", f["frameworks"])
    for e in f["evidence"][:evidence]:
        print(f"  - [{e['weight']:.2f}] {e['signal']} @ {e.get('location')}: {e['description'][:90]}")
    print(f"  ({len(f['evidence'])} evidence items)\n")


top = max(discovery["findings"], key=lambda f: f["risk"]["score"])
show(top)
print("agent_indicators:", top["metadata"].get("agent_indicators"), "| files scanned:", top["metadata"].get("files_scanned"), "\n")

mcp = next(f for f in discovery["findings"] if f["kind"] == "mcp-server")
show(mcp, evidence=2)
print("servers:", [(s["name"], s["transport"], s["url"], "inline secrets" if s["secrets_inline"] else "no inline secrets") for s in mcp["metadata"]["servers"]])

**How to read it**

- **`kind`.** `agent` only when code or a manifest establishes orchestration (`metadata.agent_indicators` counts that evidence). `framework-usage` means an SDK or protocol is used: an *enabler*, which includes implementing an MCP **server**. `mcp-server` is a client/server configuration file; `agent-config` is coding-agent configuration.
- **Confidence vs risk.** Confidence is how sure the tool is (a noisy-OR of evidence weights; `strong` ≥ 0.85 is a bucket, not "confirmed"). Risk is a sum of factors: a base per kind, capabilities (`code-exec` +15, `autonomous` +10, `data-access` +10, …), tags (`hardcoded-credential` +25, `inline-secrets` +15), MCP specifics (`mcp-stdio` +5, `mcp-auto-approve` +10, `mcp-plain-http` +10) and governance (`shadow` +25, `registered` −10, `no-owner` +10), scaled by confidence. Critical ≥ 75, high ≥ 50, medium ≥ 25.
- **Static evidence never proves that anything runs.** crewmaster is a framework *for building* agents (`agent_indicators` is 0), yet it carries `autonomous` and `code-exec`: those capabilities are what you would govern. The MCP config above declares one remote server over HTTPS with no inline secret, so none of the MCP factors fire; an `http://` URL or an inline token would change that.

*Quick check:* which two factors would push the top finding to *critical*, and what would you need to see in the repository for them to apply?

## 4 · Reconcile against an inventory

Discovery becomes governance when every agent has an **owner** and an **approval**. ShadowScan expresses that as Agent Capability Cards whose `discovery.resources` bind a card to a concrete resource id. The loop: scan with an inventory → `inventory stubs` writes one card skeleton per shadow finding → a human completes a stub and moves it into the inventory → rescan → `diff`.

Start with an **explicitly empty** inventory. That is not the same as no inventory: it switches reconciliation on, so every finding gets a real verdict. Nothing in the code changes; only the question does.

In [ ]:
(LAB / "inventory" / "agents.yaml").write_text("agents: []\n", encoding="utf-8")
shadowscan("code", *PATHS, "--inventory", LAB / "inventory", "--format", "json", "--output", LAB / "reports/unregistered.json")
unregistered = load("unregistered")
assert all(f["shadow"] is True for f in unregistered["findings"]), "with an empty inventory every finding is unmatched"

before = {f["id"]: f["risk"]["score"] for f in discovery["findings"]}
print("\nrisk before -> after adding the empty inventory (shadow +25)")
for f in sorted(unregistered["findings"], key=lambda f: -f["risk"]["score"]):
    print(f"  {before[f['id']]:3} -> {f['risk']['score']:3} {f['risk']['level']:8} {f['title'][:70]}")

Now generate the card stubs. **Write them outside `inventory/`**: inventory paths are searched recursively, and an unreviewed stub inside the inventory would silently count as an approval. Then make one decision: the MCP client configuration (`.mcp.json`) is sanctioned and owned by *Platform-Security*. Everything else stays unmatched on purpose.

In [ ]:
pending = LAB / "inventory_pending"  # deliberately NOT inside inventory/
shutil.rmtree(pending, ignore_errors=True)
shadowscan("inventory", "stubs", LAB / "reports/unregistered.json", "--out", pending, "--min-risk", "medium")
stub = next(p for p in sorted(pending.glob("*.yaml")) if "/.mcp.json" in p.read_text(encoding="utf-8"))
print(stub.read_text(encoding="utf-8"))

# The decision: complete the stub (owner) and move it into the inventory. It binds exactly one resource and one observation type.
approved = LAB / "inventory" / "approved" / "mcp-client-config.yaml"
approved.parent.mkdir(exist_ok=True)
approved.write_text(stub.read_text(encoding="utf-8").replace("owner_team: UNKNOWN", "owner_team: Platform-Security"), encoding="utf-8")
_ = shadowscan("inventory", "check", LAB / "inventory")

In [ ]:
shadowscan("code", *PATHS, "--inventory", LAB / "inventory", "--format", "json", "--output", LAB / "reports/reconciled.json")
reconciled = load("reconciled")
registered = [f for f in reconciled["findings"] if f["shadow"] is False]
assert len(registered) == 1 and registered[0]["owner"] == "Platform-Security", "exactly the approved resource must be registered"
print()
show(registered[0], evidence=0)
_ = shadowscan("diff", LAB / "reports/unregistered.json", LAB / "reports/reconciled.json")

**Validated result:** the `.mcp.json` finding flips to `shadow: false`, inherits the owner, and drops from **59 to 15** as `shadow(+25)` and `no-owner(+10)` are replaced by `registered(−10)`. `diff` shows it under *changed* with the fields that moved. The lower score reflects a governance decision, not a code fix.

Three traps to remember: a broad pattern such as `github:*` is an explicit broad approval of everything it matches; names in a card only produce suggestions and never approve; two cards matching one resource leave it **unregistered** as ambiguous.

## 5 · Report and gate

`--format html` gives a self-contained, filterable report (`sarif` feeds GitHub code scanning, `markdown` and `csv` exist too). `--fail-on high` turns the heuristic level into exit code **2**, which is how a pipeline fails the job. It is a triage convenience, not a production control: an incomplete scan still exits 3, and the project's own CI example leaves the gate off until a threshold has been validated for the repository.

In [ ]:
rc = shadowscan("code", *PATHS, "--inventory", LAB / "inventory", "--fail-on", "high",
                "--format", "html", "--output", LAB / "reports/shadow-agent-scan.html")
print({0: "PASS: nothing at or above high", 2: "FAIL: unregistered high-risk findings need review", 3: "INCONCLUSIVE: scan incomplete"}.get(rc, "ERROR"))
print("open in a browser:", LAB / "reports/shadow-agent-scan.html")

## Done

You installed a pinned scanner, discovered 12 agent-related findings in two repositories without running their code, explained a score from its factors, turned discovery into a governance decision with one exact approval, proved the effect with `diff`, and produced a report and a gate result.

**Next steps**

- Live connectors: `shadowscan code --github-repo owner/name` or a YAML config with `code.gitlab` and `projects: [group/project]`; without a token the credential-name listings are denied and the scan exits 3 with the code findings intact. See [code connectors](https://github.com/aisecnomad/Project-Nexus/blob/9d4a6f1261f1daba97c2c147ca088dec78e89a46/docs/connectors/code.md).
- CI: copy [`examples/github-action-code-scan.yml`](https://github.com/aisecnomad/Project-Nexus/blob/9d4a6f1261f1daba97c2c147ca088dec78e89a46/examples/github-action-code-scan.yml), which pins the scanner by commit SHA, uploads SARIF, and keeps the gate off until you validate a threshold.
- Deeper reading: [inventory and reconciliation](https://github.com/aisecnomad/Project-Nexus/blob/9d4a6f1261f1daba97c2c147ca088dec78e89a46/docs/inventory.md) · [risk scoring](https://github.com/aisecnomad/Project-Nexus/blob/9d4a6f1261f1daba97c2c147ca088dec78e89a46/docs/concepts/risk.md) · [scan state and `diff`](https://github.com/aisecnomad/Project-Nexus/blob/9d4a6f1261f1daba97c2c147ca088dec78e89a46/docs/scanning.md).

*Validated 3 Oct 2026 by executing every cell against both pinned checkouts with ShadowScan 0.1.1 at the pinned commit (Linux x86_64, Python 3.11). The example repositories are public projects whose maintainers are not affiliated with this lab.*